# MMS-300M Bengali dialect MoE — resumable T4×2 training

Attach the processed private Vaani Dataset. For later sessions, also attach the previous run checkpoint Dataset and set `PRIOR_RUN_DIR`. Keep Internet enabled so the first session can retrieve MMS-300M.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/diyalibiswas1998/bengali-dialect-asr.git"
REPO_DIR = Path("/kaggle/working/bengali-dialect-asr")
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


In [ ]:
# Edit these paths to match the attached Kaggle Dataset slugs.
DATASET_DIR = Path("/kaggle/input/vaani-bengali-processed")
PRIOR_RUN_DIR = None  # Example: Path("/kaggle/input/bengali-moe-checkpoints/moe-run")
RUN_DIR = Path("/kaggle/working/moe-run")
EXPERIMENT = "moe"  # baseline, moe, top1, no_dialect, or no_shared

if not (DATASET_DIR / "metadata.json").exists():
    raise FileNotFoundError(f"Processed dataset not found at {DATASET_DIR}")
if PRIOR_RUN_DIR and not RUN_DIR.exists():
    shutil.copytree(PRIOR_RUN_DIR, RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"experiment={EXPERIMENT} data={DATASET_DIR} output={RUN_DIR}")


In [ ]:
# Mandatory two-process forward/backward and exact state-restoration check.
subprocess.check_call([
    "accelerate", "launch", "--config_file", str(REPO_DIR / "configs/accelerate_t4x2.yaml"),
    str(REPO_DIR / "scripts/smoke_test_research.py"),
    "--config", str(REPO_DIR / "configs/research.yaml"),
    "--data-dir", str(DATASET_DIR), "--require-two-gpus",
])


In [ ]:
command = [
    "accelerate", "launch", "--config_file", str(REPO_DIR / "configs/accelerate_t4x2.yaml"),
    str(REPO_DIR / "scripts/train_research.py"),
    "--config", str(REPO_DIR / "configs/research.yaml"),
    "--data-dir", str(DATASET_DIR),
    "--output-dir", str(RUN_DIR),
    "--experiment", EXPERIMENT,
]
if list(RUN_DIR.glob("checkpoint-*")):
    command += ["--resume", "latest"]
subprocess.check_call(command)


In [ ]:
# Evaluate only when all three phases are present.
final_checkpoint = RUN_DIR / "checkpoint-phase-3"
if final_checkpoint.exists():
    subprocess.check_call([
        sys.executable, str(REPO_DIR / "scripts/validate_checkpoint.py"),
        "--checkpoint", str(final_checkpoint), "--expected-processes", "2",
    ])
    subprocess.check_call([
        "accelerate", "launch", "--config_file", str(REPO_DIR / "configs/accelerate_t4x2.yaml"),
        str(REPO_DIR / "scripts/evaluate_research.py"),
        "--checkpoint", str(final_checkpoint), "--data-dir", str(DATASET_DIR),
    ])
else:
    print("This session ended before phase 3. Save RUN_DIR as a private Kaggle Dataset version and resume next session.")


After every Kaggle session, save `RUN_DIR` as a new private Dataset version. Keep separate run directories for the baseline, main MoE, and each ablation. The split fingerprints in every `config.json` must match before comparing results.
